# 4. Dividir janelas reais sem vazamento de indivíduos

Dividir aleatoriamente as janelas em decodificadores de EEG entre sujeitos após o conjunto de treinamento atinge uma precisão próxima a 99% e falham em um participante excluído. Isso pois cada gravação produz centenas de janelas sobrepostas do mesmo cérebro, então um embaralhamento uniforme espalha cada sujeito entre tanto o conjunto de treinamento quanto o de teste, e o modelo memoriza impressões digitais cerebrais de cada sujeito (frequência cardíaca, amplitude alpha, impedância do eletrodo) em vez da tarefa que queremos realmente decodificar.

Vamos ver primeiro a falha em janelas sintéticas e, em seguida, vamos reconstruir o split com `eegdash.splits` e um splitter `GroupKFOld-flavoured cross-subject`. A figura fina coloca ambas as estratégias lado a lado.

## 4.1 Por que o vazamento de sujeito ataca o EEG mais profundamente do que outros domínios?

As características específicas de cada sujeito dominam qualquer janela EEG. Largura do crânio, colocação dos eletrodos, condutividade capilar e frequência cardíaca desempenham como uma "assinatura" dos pacientes. 

Brookshire et. al. 2024 quantificou isso em 81 EEGs clínicos em papers de deep-learning: Quando sujeitos aparecem em ambos os lados de um split, a acurácia foi de 0.83, mas em sujeitos que não aparecem em ambos os lados de um split, a mesma arquitetura designava uma acurácia de 0.62. Metade dos estudos avaliados vazou.

Cada janela mantém a assinatura de cada paciente. O conserto disso é estrutural: separamos sujeitos, não janelas. Agrupamos toda janela pelo id do sujeito e deixamos `sklearn.model_selection.GroupKFold` colocar cada sujeito em exatamente um caso de teste.

Para validar o resultado:

- **Leakage Check**. Depois do split, rode `assert_no_leakaage(train_ids, test_ids)`. Isso não deve retornar erro e retornar `True`.
- **Leakage Report**. O JSON `leakage_report` deve mostrar zero.
- **Accuracy Gap**. É esperado que o naive random split performe uma taxa de vazamento de 10-30 pontos.

## 4.2 Requisitos

Definir a seed de np mantém o embaralhamento do shuffle e a ordem reprodutível.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import eegdash
from collections import Counter

from moabb.evaluations.splitters import CrossSessionSplitter, CrossSubjectSplitter
from sklearn.model_selection import GroupKFold
from eegdash.viz import use_eegdash_style

use_eegdash_style()
warnings.simplefilter("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)
print(f"eegdash {eegdash.__version__}; numpy {np.__version__}")

/home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


eegdash 0.9.1; numpy 2.5.3


## 4.3 Construir uma tabela de metadados de janela para 12 sujeitos

Temos uma simulação de um dataset aqui (metadados experimentais).

Temos 12 sujeitos com 2 sessões cada. Cada sessão possui 8 janelas. Então, o número total de janelas é 192.



In [ ]:
N_SUBJECTS = 12
N_SESSIONS = 2
N_WINDOWS = 8

# Experimental Metadata
rows = [
    {
        "subject": f"sub-{s:02d}",
        "session": f"ses-{ses:02d}",
        "run": "run-01",
        "dataset": "ds-windowed-tutorial",
        "sample_id": f"sub-{s:02d}__ses-{ses:02d}__w{w:03d}",
        "target": int((s + w) % 2), # 0/1
    }
    for s in range(1, N_SUBJECTS + 1)
    for ses in range(1, N_SESSIONS + 1)
    for w in range(N_WINDOWS)
]
raw_metadata = pd.DataFrame(rows)

# For a Braindecode concat-of-WindowsDataset the canonical metadata
# accessor is :meth:`braindecode.datasets.BaseConcatDataset.get_metadata`
# (one row per window, BIDS columns from each record's ``description``
# already merged in). When you start from a hand-built DataFrame, just
# use it directly: pass the frame as ``metadata`` and the column you
# want to stratify on as ``y``.
metadata = raw_metadata
y = metadata["target"].to_numpy()

pd.Series(
    {
        "rows": len(metadata),
        "subjects": metadata["subject"].nunique(),
        "sessions": metadata["session"].nunique(),
        "y dtype": str(y.dtype),
        "class 0 / class 1": (
            f"{int((metadata.target == 0).sum())} / {int((metadata.target == 1).sum())}"
        ),
    },
    name="value",
).to_frame()

,value
rows,192
subjects,12
sessions,2
y dtype,int64
class 0 / class 1,96 / 96


## 4.4 Separando janelas do jeito ERRADO

**Predict**. Se embaralharmos as 192 janelas uniformemente e colocarmos 20% no grupo de teste, quantos sujeitos vão acabar em AMBOS (treino e teste)?

**Run**. Um embaralhamento de janelas aleatórias.

In [ ]:
rng = np.random.default_rng(SEED) # random number generator
shuffled = rng.permutation(len(metadata)) # Creates shuffled array idx
cut = int(0.8 * len(metadata)) # Select 80% for training
naive_train = metadata.iloc[shuffled[:cut]] # Windows for train (80%)
naive_test = metadata.iloc[shuffled[cut:]] # Windows for test (20%)

# True/False if leaked
leaked = sorted(set(naive_train["subject"]) & set(naive_test["subject"]))
naive_overlap = len(leaked)

pd.Series(
    {
        "train rows": len(naive_train),
        "test rows": len(naive_test),
        "subjects in train": naive_train["subject"].nunique(),
        "subjects in test": naive_test["subject"].nunique(),
        "subject_overlap": f"{naive_overlap} / {N_SUBJECTS}",
    },
    name="value",
).to_frame()

,value
train rows,153
test rows,39
subjects in train,12
subjects in test,11
subject_overlap,11 / 12


## 4.5 Gerando janelas sem vazamento

rodar `get_splitter` retorna uma instância de `CrossSubjectSplitter` do MOAB. Aqui, usamos N_FOLDS=5. Assim, ao invés de testar um sujeito por vez (*Leave-One-Group-Out*), agrupamos os sujeitos em 5 partições para acelerar o processo de auditoria/teste.

`splitter.split(y, metadata)` retorna os índices de treino e teste (`(tr_idx, te_idx)`) para cada um dos 5 folds. Se um sujeito for para o conjunto de teste de um fold, todas as amostras dele estarão no teste e nenhuma no treino daquele mesmo fold. Ele faz isso investigando os metadados (`metadata["subject"]`).

In [ ]:
N_FOLDS = 5
# ``cv_class=GroupKFold`` swaps MOABB's default ``LeaveOneGroupOut`` for
# a parametrisable fold count so the audit stays short. Without it you
# get LeaveOneGroupOut, which produces one fold per subject.
splitter = CrossSubjectSplitter(cv_class=GroupKFold, n_splits=N_FOLDS)
y = metadata["target"].to_numpy() # Gets targets
n_rows = len(metadata) # Gets number of rows
folds: list[tuple[np.ndarray, np.ndarray]] = []

# Using splitter intance to split
for tr_idx, te_idx in splitter.split(y, metadata):
    tr_mask = np.zeros(n_rows, dtype=bool)
    tr_mask[tr_idx] = True
    te_mask = np.zeros(n_rows, dtype=bool)
    te_mask[te_idx] = True
    folds.append((tr_mask, te_mask))
    
pd.Series(
    {
        "splitter_class": type(splitter).__name__,
        "n_folds": len(folds),
        "target": "target",
        "random_seed": SEED,
    },
    name="value",
).to_frame()

,value
splitter_class,CrossSubjectSplitter
n_folds,5
target,target
random_seed,42


## 4.6 Provando que não há vazamento

`assert_ no_leakage` caminha por todo fold, verifica a intersecção dos valores `subject` entre treino e test, e imprime uma linha JSON:

```text
{"leakage_report": {"overlap": 0, "by": "subject"}}
```

Um split limpo imprime `overlap:0`, e uma split com vazamento produz um valor diferente de zero + `LeakageError`.

In [ ]:
# Verifies intersections across train/test in each folder
overlap = max(
    len(set(metadata.loc[tr, "subject"]) & set(metadata.loc[te, "subject"]))
    for tr, te in folds
)
sys.stdout.write(
    json.dumps({"leakage_report": {"overlap": int(overlap), "by": "subject"}}) + "\n"
)
sys.stdout.flush()
assert overlap == 0, "Cross-subject split leaked!"

# Inline per-fold audit; matches the shape of the deprecated
# ``describe_split`` summary so the rest of the tutorial reads the same.
per_fold = []
for tr_mask, te_mask in folds:
    train = metadata.loc[tr_mask]
    test = metadata.loc[te_mask]
    per_fold.append(
        {
            "n_train": len(train),
            "n_test": len(test),
            "subjects_train": train["subject"].nunique(),
            "subjects_test": test["subject"].nunique(),
            "class_balance_train": dict(Counter(train["target"].dropna().tolist())),
            "class_balance_test": dict(Counter(test["target"].dropna().tolist())),
        }
    )
fold0 = per_fold[0]
balance0 = fold0["class_balance_test"]
class_balance_ratio = max(balance0.values()) / (sum(balance0.values()) or 1)
pd.Series(
    {
        "fold": 0,
        "subjects_train": fold0["subjects_train"],
        "subjects_test": fold0["subjects_test"],
        "n_train": fold0["n_train"],
        "n_test": fold0["n_test"],
        "class_balance_test": dict(balance0),
        "class_balance_ratio": round(float(class_balance_ratio), 3),
    },
    name="value",
).to_frame()

{"leakage_report": {"overlap": 0, "by": "subject"}}


,value
fold,0
subjects_train,9
subjects_test,3
n_train,144
n_test,48
class_balance_test,"{0: 24, 1: 24}"
class_balance_ratio,0.5


In [ ]:
audit_df = pd.DataFrame(per_fold)
audit_df.insert(0, "fold", range(len(audit_df)))
audit_df[
    [
        "fold",
        "n_train",
        "n_test",
        "subjects_train",
        "subjects_test",
        "class_balance_train",
        "class_balance_test",
    ]
]

,fold,n_train,n_test,subjects_train,subjects_test,class_balance_train,class_balance_test
0,0,144,48,9,3,"{1: 72, 0: 72}","{0: 24, 1: 24}"
1,1,144,48,9,3,"{0: 72, 1: 72}","{1: 24, 0: 24}"
2,2,160,32,10,2,"{1: 80, 0: 80}","{1: 16, 0: 16}"
3,3,160,32,10,2,"{1: 80, 0: 80}","{0: 16, 1: 16}"
4,4,160,32,10,2,"{1: 80, 0: 80}","{1: 16, 0: 16}"


## 4.7 Resultados

In [ ]:
print(
    "Final invariants:",
    json.dumps(
        {
            "n_subjects_total": int(metadata["subject"].nunique()),
            "n_folds": int(len(folds)),
            "subject_overlap": int(overlap),
            "naive_random_split_overlap": int(naive_overlap),
            "class_balance_ratio_fold0": round(float(class_balance_ratio), 3),
        }
    ),
)

Final invariants: {"n_subjects_total": 12, "n_folds": 5, "subject_overlap": 0, "naive_random_split_overlap": 11, "class_balance_ratio_fold0": 0.5}


## 4.8 Split entre sessões, não sujeitos

Basta modificar a classe utilizada.

In [ ]:
session_splitter = CrossSessionSplitter(cv_class=GroupKFold, n_splits=2)
session_folds: list[tuple[np.ndarray, np.ndarray]] = []
for tr_idx, te_idx in session_splitter.split(y, metadata):
    tr_mask = np.zeros(n_rows, dtype=bool)
    tr_mask[tr_idx] = True
    te_mask = np.zeros(n_rows, dtype=bool)
    te_mask[te_idx] = True
    session_folds.append((tr_mask, te_mask))
session_overlap = max(
    len(set(metadata.loc[tr, "session"]) & set(metadata.loc[te, "session"]))
    for tr, te in session_folds
)
print(f"cross_session overlap: {session_overlap}")

cross_session overlap: 0
